In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import arya
import pandas as pd

In [ ]:
import ana_utils

In [ ]:
from astropy import units as u

In [ ]:
from dataclasses import dataclass

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
obj = "yasone3"

In [ ]:
cat = ana_utils.read_catalogue(obj, filter_bad=False)

In [ ]:
obs_props = ana_utils.get_obs_props(obj)
A_g, A_r, A_i = ana_utils.get_extinction(obs_props["A_V"])

In [ ]:
plt.scatter(cat["xi"], cat["eta"], s=1)
plt.xlim(-3, 3)
plt.ylim(-3, 3)

In [ ]:
cat.columns

In [ ]:
def calibrate_mag_col(cat, col):
    for filt in ["G", "R", "I"]:
        filt_good = cat[f"{filt}_MAG"] > 19
        filt_good &= cat[f"{filt}_MAG"] < 22
        filt_good &= ~cat[f"{filt}_MAG"].mask
        if hasattr(cat[f"{filt}_{col}"], "mask"):
            filt_good &= ~cat[f"{filt}_{col}"].mask
            
        residual = np.median((cat[f"{filt}_{col}"] - cat[f"{filt}_MAG"])[filt_good])

        print(residual)
        cat[f"{filt}_{col}"] -= residual

In [ ]:
plt.scatter(cat["R_MAG_APER_0"], cat["R_MAG_APER_0"] - cat["R_MAG"], s=1)

In [ ]:
mag_choices = ["MAG_APER_0", "MAG_APER_1", "MAG_APER_2", "MAG_APER_3", "MAG_APER_4", "MAG_APER_5", "MAG_APER_6", "MAG_AUTO", "MAG_WIN", "MAG_ISO"]

In [ ]:
for mag in mag_choices:
    calibrate_mag_col(cat, mag)

In [ ]:
col = "MAG_WIN"
for filt in ["G", "R", "I"]:
    plt.scatter(cat[f"{filt}_{col}"], cat[f"{filt}_{col}"] - cat[f"{filt}_MAG"], s=1)
    plt.axhline(0, color="k")
    plt.ylim(-1, 1)
    plt.xlim(12, 30)
    plt.show()

In [ ]:
def count_centre(cat, radius, xi0=0, eta0=0):
    return np.sum((cat["xi"] - xi0)**2 + (cat["eta"] - eta0)**2 < radius**2)

In [ ]:
def count_not_centre(cat, radius, xymax=3, xi0=0, eta0=0):
    filt = (cat["xi"] - xi0)**2 + (cat["eta"] - eta0)**2 >= radius**2
    filt &= cat["xi"] < xymax
    filt &= cat["xi"] > -xymax
    filt &= cat["eta"] < xymax
    filt &= cat["eta"] > -xymax

    return np.sum(filt)

In [ ]:
Ntot = np.sum((cat["xi"] > -3) & (cat["xi"] < 3) & (cat["eta"] < 3) & (cat["eta"] > -3))
Ncen = count_centre(cat, 40/60)

In [ ]:
cat_cen = ana_utils.select_centre(cat, ana_utils.get_coord0(obj), 40 * u.arcsec)
assert count_not_centre(cat, 40/60) == Ntot - Ncen
assert len(cat_cen) == Ncen

In [ ]:
count_centre(cat, 40/60)

In [ ]:
def counts_to_stats(Ntot, Ncen, area_scale):
    Ntot_normed = Ntot * area_scale
    Ncen_err = np.sqrt(Ncen)
    Ntot_normed_err = np.sqrt(Ntot) * area_scale

    num_err = np.sqrt(Ntot_normed_err**2 + Ncen_err**2)

    sigma = (Ncen - Ntot_normed) / Ntot_normed
    sigma_err = np.sqrt((Ncen_err / Ntot_normed)**2 + (Ncen/Ntot_normed**2 * Ntot_normed_err)**2)

    return sigma, sigma_err, Ntot, Ncen

In [ ]:
def inner_overdensity(cat, radius, xymax=3):
    area_tot = (2*xymax)**2
    area_cen = np.pi * radius**2
    area_scale = area_cen / (area_tot - area_cen)

    Ntot = count_not_centre(cat, radius, xymax=xymax)
    Ncen = count_centre(cat, radius)

    return counts_to_stats(Ntot, Ncen, area_scale)


In [ ]:
import read_iso

In [ ]:
isos_fe_h = ["m1.00", "m1.25", "m1.50", "m1.75", "m2.00", "m2.50", "m3.00"]

isonames = [f"../MIST/MIST_v1.2_vvcrit0.4_SDSSugriz/MIST_v1.2_feh_{iso_fe_h}_afe_p0.0_vvcrit0.4_SDSSugriz.iso.cmd" for iso_fe_h in isos_fe_h]
isochrones = {fe_h: read_iso.ISOCMD(name) for fe_h, name in zip(isos_fe_h, isonames)}


In [ ]:
def make_polygon(iso, mag_err, dm, A_b, A_r, iso_width=0.05, b="SDSS_g", r="SDSS_r", err_scale=1):
    filt = iso["phase"] < 3

    x = iso[b][filt].data - iso[r][filt].data + A_b - A_r
    y = iso[b][filt].data + dm + A_b

    x_min = x - err_scale * mag_err(y) - iso_width
    x_max = x + err_scale * mag_err(y) + iso_width

    return np.hstack([x_min, x_max[::-1], x_min[0]]), np.hstack([y, y[::-1], y[0]])

In [ ]:
gr_err = ana_utils.fit_err(cat["G_MAG"], cat["GR_ERR"])
ri_err = ana_utils.fit_err(cat["R_MAG"], cat["RI_ERR"])

In [ ]:
def select_gr(cat_gr, params):
    iso = isochrones[params.iso_fe_h][params.iso_log_age]

    A_g, A_r, A_i = ana_utils.get_extinction(params.A_V)
    x_poly_gr, y_poly_gr = make_polygon(iso, gr_err, params.dm, A_b=A_g, A_r=A_r, r="SDSS_r", b="SDSS_g", iso_width=params.iso_width, err_scale=params.err_scale)
    
    cmd_filt_gr = ana_utils.is_in_poly(cat_gr[params.gcol].data - cat_gr[params.rcol].data, cat_gr[params.gcol].data, x_poly_gr, y_poly_gr)

    return cmd_filt_gr



def select_ri(cat, params):
    iso = isochrones[params.iso_fe_h][params.iso_log_age]

    A_g, A_r, A_i = ana_utils.get_extinction(params.A_V)
    x_poly_gr, y_poly_gr = make_polygon(iso, ri_err, params.dm, 
                                        A_b=A_r, A_r=A_i, r="SDSS_i", b="SDSS_r", iso_width=params.iso_width, err_scale=params.err_scale)
    
    cmd_filt = ana_utils.is_in_poly(cat[params.rcol].data - cat[params.icol].data, cat[params.rcol].data, x_poly_gr, y_poly_gr)

    return cmd_filt

In [ ]:
@dataclass
class filter_params:
    r_cen: float = 40/60
    xymax: float = 3
    flags_max: int = 4
    flags_weight_max: int = 2
    detection_sigma: float = 1.5
    A_V: float = obs_props["A_V"]
    dm: float = obs_props["dm"]
    iso_width: float = 0
    iso_fe_h: str = "m2.00"
    iso_log_age: float = 10.1
    err_scale: float = 1
    mode: str = "gr"
    gcol:str = "G_MAG"
    rcol: str = "R_MAG"
    icol: str = "I_MAG"


In [ ]:
filter_params()

In [ ]:
def filter_nans(col):
    filt = np.isfinite(col)
    if hasattr(col, "mask"):
        filt &= ~col.mask

    return filt


In [ ]:
def filter_catalog(cat, params):    
    filt = cat["R_FLAGS"] <= params.flags_max
    filt &= cat["R_FLAGS_WEIGHT"] <= params.flags_weight_max
    filt &= cat["R_SNR"] > params.detection_sigma

    if params.mode == "gr":
        filt &= select_gr(cat, params)
        filt &= filter_nans(cat[params.gcol])
        filt &= filter_nans(cat[params.rcol])
    elif params.mode == "ri":
        filt &= select_ri(cat, params)
        filt &= filter_nans(cat[params.icol])
        filt &= filter_nans(cat[params.rcol])
    elif params.mode == "n":
        pass
    elif params.mode == "r":
        filt &= filter_nans(cat[params.rcol])
    else:
        raise Exception(f"mode not known {params.mode}")
        return None


    return cat[filt]


In [ ]:
from dataclasses import asdict

In [ ]:
asdict(filter_params())

In [ ]:
from scipy.spatial import cKDTree

In [ ]:
def background_sample_counts(cat, radius, xymax=3, N=1000):
    points = cat[["xi", "eta"]].to_pandas().to_numpy()
    tree = cKDTree(points)

    # sampling radii from outside the central region but within xy max
    xi = np.random.uniform(-xymax+radius, -1.5*radius, N) * np.random.choice([1, -1], N)
    eta = np.random.uniform(-xymax+radius, -1.5*radius, N) * np.random.choice([1, -1], N)

    centres = np.column_stack([xi, eta])
    counts = tree.query_ball_point(centres, r=radius, return_length=True)
    return counts

In [ ]:
from astropy.stats import mad_std

In [ ]:
def catalog_stats(cat, 
        params
    ):

    cat_filtered = filter_catalog(cat,
        params
    )
    sigma, sigma_err, Ntot, Ncen = inner_overdensity(cat_filtered, params.r_cen, xymax=params.xymax)



    N_bkgs = background_sample_counts(cat_filtered, params.r_cen, xymax=params.xymax)
    
    area_tot = (2*params.xymax)**2
    area_cen = np.pi * params.r_cen**2
    area_scale = area_cen / (area_tot - area_cen)
    Nbkg_med = np.median(N_bkgs)
    Nbkg_std = mad_std(N_bkgs)

    delta = (Ncen - Nbkg_med) / Nbkg_med
    delta_err = np.sqrt((np.sqrt(Ncen) / Nbkg_med)**2 + (Ncen/Nbkg_med**2 * Nbkg_std)**2)

    pval = np.mean(N_bkgs > Ncen)

    return pd.DataFrame(dict(
        pval = pval,
        sigma = sigma, 
        sigma_err = sigma_err,
        delta = delta,
        delta_err = delta_err,
        Ntot = Ntot,
        Ncen = Ncen,
        Nbkg_med = Nbkg_med,
        Nbkg_std = Nbkg_std,
        **asdict(params)
        
    ), index=[0])

In [ ]:
catalog_stats(cat, filter_params())

# Running the MCMC samples

In [ ]:
df = pd.DataFrame()

for i in range(300):

    if i % 20 == 0:
        print("step ", i)

    col = np.random.choice(mag_choices)
    params = filter_params(
        detection_sigma = np.random.choice([1.5, 2, 2.5, 3, 6]),
        r_cen = np.random.uniform(10, 80)/60,
        A_V = obs_props["A_V"] + np.random.normal(0, 0.2),
        dm = obs_props["dm"] + np.random.normal(0, 0.2),
        flags_max = np.random.choice([0, 4, 8, 1024]),
        flags_weight_max = np.random.choice([0, 2, 1024]),
        iso_fe_h = np.random.choice(isos_fe_h),
        iso_log_age = np.random.choice(np.arange(9.4, 10.3, 0.1)),
        iso_width = np.random.uniform(0, 0.1),
        mode = np.random.choice([ "ri",  "gr", "n"]),
        gcol = f"G_{col}",
        rcol = f"R_{col}",
        icol = f"I_{col}",
    )
    d = catalog_stats(cat, params)

    df = pd.concat([df, d], ignore_index=True)

In [ ]:
df["significance"] = df["delta"] / df["delta_err"]
df.loc[~np.isfinite(df["significance"]), "significance"] = 0

In [ ]:
plt.scatter(df["significance"], df["pval"], c=np.log10(df["delta"]))

plt.yscale('log')
plt.gca().invert_yaxis()

plt.ylabel("pvalue")
plt.xlabel("significance")
plt.colorbar(label="delta (overdensity)")

In [ ]:
plt.scatter(df["Ncen"], df["significance"], c=np.log10(df["pval"]))

plt.xscale('log')

plt.ylabel("contrast ")
plt.xlabel("number of stars in centre")
plt.colorbar(label="pvalue")

In [ ]:
for col in df.columns:
    if col not in ["sigma", "sigma_err", "Ncen", "Ntot", "Nbkg_med", "Nbkg_std", "pval", "delta", "delta_err", "significance"]:
        plt.figure(figsize=(4, 2))
        if len(df[col].unique()) == 1:
            continue
        elif len(df[col].unique()) < 20:
            groups = df.groupby(col).groups
            plt.boxplot([df.iloc[idx]["significance"] for idx in groups.values()], tick_labels=groups.keys(), )
            plt.xlabel(col)
            plt.ylabel("significance")
            plt.xticks(rotation=90)

        else:
            plt.scatter(df[col], df["significance"], c=-np.log10(df["pval"]), s=3)
            plt.xlabel(col)
            plt.ylabel("significance")
        # plt.gca().invert_yaxis()
        # plt.yscale("log")
            plt.colorbar(label="-log pvalue")
        plt.show()

In [ ]:
def retrieve_params(df_row):
    kwargs = {key: df_row[key] for key in filter_params.__dataclass_fields__.keys()}

    return filter_params(**kwargs)

In [ ]:
df.sort_values("significance")

In [ ]:
plt.hist(np.log10(df["pval"][df.pval > 0]))

In [ ]:
params_best = retrieve_params(df.iloc[np.argmax(df.significance)])

In [ ]:
params_best

In [ ]:
catalog_stats(cat, params_best)

# Better plots

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
def plot_riso(cat, params, xi=-1.5/60, eta=-1.5/60, ra0=obs_props["ra"], dec0=obs_props["dec"]):

    cat_filtered = filter_catalog(cat, params)
    
    # xi = np.random.uniform(1.5 *params.r_cen, 3 - params.r_cen) / 60 * np.random.choice([-1, 1])
    # eta = np.random.uniform(1.5 * params.r_cen, 3 - params.r_cen) / 60 * np.random.choice([-1, 1])
    
    coord1 = SkyCoord(ra=ra0 + xi/np.cos(np.deg2rad(dec0)), dec=dec0 + eta, unit=u.deg)
    coord0 = SkyCoord(ra=ra0, dec=dec0, unit=u.deg)

    df_bkg = ana_utils.select_centre(cat_filtered, coord1, params.r_cen * u.arcmin)
    df_cen = ana_utils.select_centre(cat_filtered, coord0, params.r_cen * u.arcmin)
    df_bkg_all = ana_utils.select_centre(cat, coord1, params.r_cen * u.arcmin)
    df_cen_all = ana_utils.select_centre(cat, coord0, params.r_cen * u.arcmin)

    fig, axs = plt.subplots(2, 3, figsize=(6, 4))

    plt.sca(axs[0][0])
    tangent_axis()
    plt.scatter(cat["xi"], cat["eta"], s=1, lw=0, color="k")
    
    circ = plt.Circle((0, 0), params.r_cen, color="C0", lw=1, fill=False)
    plt.gca().add_patch(circ)
    circ = plt.Circle((xi*60, eta*60), params.r_cen, color="C1", lw=1, fill=False)
    plt.gca().add_patch(circ)
    
    
    plt.title("all")

    plt.sca(axs[1][0])
    tangent_axis()
    plt.scatter(cat_filtered["xi"], cat_filtered["eta"], s=1, lw=0, color="C0")
    
    circ = plt.Circle((0, 0), params.r_cen, color="C0", lw=1, fill=False)
    plt.gca().add_patch(circ)
    circ = plt.Circle((xi*60, eta*60), params.r_cen, color="C1", lw=1, fill=False)
    plt.gca().add_patch(circ)
    
    plt.title("selected")

    plt.sca(axs[1][1])
    gr_axis()
    plot_iso_gr(params)
    
    x = df_bkg_all[params.gcol] - df_bkg_all[params.rcol]
    y = df_bkg_all[params.gcol]
    plt.scatter(x, y, lw=0, s=1, color="k")
    
    x = df_bkg[params.gcol] - df_bkg[params.rcol]
    y = df_bkg[params.gcol]
    plt.scatter(x, y, lw=0, s=3, color="C0")

    plt.title("background")

    
    plt.sca(axs[0][1])
    gr_axis()
    plot_iso_gr(params)

    x = df_cen_all[params.gcol] - df_cen_all[params.rcol]
    y = df_cen_all[params.gcol]
    plt.scatter(x, y, lw=0, s=1, color="k")

    x = df_cen[params.gcol] - df_cen[params.rcol]
    y = df_cen[params.gcol]
    plt.scatter(x, y, lw=0, s=3, color="C0")

    plt.title("centre")

    
    plt.sca(axs[1][2])
    ri_axis()
    plot_iso_ri(params)
    x = df_bkg_all[params.rcol] - df_bkg_all[params.icol]
    y = df_bkg_all[params.rcol]
    plt.scatter(x, y, lw=0, s=1, color="k")
    
    x = df_bkg[params.rcol] - df_bkg[params.icol]
    y = df_bkg[params.rcol]
    plt.scatter(x, y, lw=0, s=3, color="C0")
    plt.title("background")


    plt.sca(axs[0][2])
    ri_axis()
    plot_iso_ri(params)
    
    x = df_cen_all[params.rcol] - df_cen_all[params.icol]
    y = df_cen_all[params.rcol]
    plt.scatter(x, y, lw=0, s=1, color="k")

    x = df_cen[params.rcol] - df_cen[params.icol]
    y = df_cen[params.rcol]
    plt.scatter(x, y, lw=0, s=3, color="C0")
    
    plt.title("centre")

    # plot_ri_depth()

    plt.tight_layout()

In [ ]:
def tangent_axis():
    plt.ylabel(r"$\eta/\textrm{arcmin}$")
    plt.xlabel(r"$\xi/\textrm{arcmin}$")
    plt.xlim(3, -3)
    plt.ylim(-3, 3)
    plt.xticks(np.arange(-3, 4))
    plt.yticks(np.arange(-3, 4))
    plt.gca().set_aspect(1)

In [ ]:
def ri_axis():
    plt.xlabel(r"$r-i$ (mag)")
    plt.ylabel(r"$r$ (mag)")

    plt.xlim(-1.2, 1.8)
    plt.ylim(27, 17)
    plt.gca().set_box_aspect(1)


def gr_axis():
    plt.xlabel(r"$g-r$ (mag)")
    plt.ylabel(r"$g$ (mag)")
    plt.xlim(-1, 2)
    plt.ylim(27, 17)
    plt.gca().set_box_aspect(1)


In [ ]:
def plot_iso(iso, dm, A_b, A_r, mag_err, phase_max=3, b="SDSS_g", r="SDSS_r", iso_width=0):
    filt = iso["phase"] < phase_max
    plt.plot(iso[b][filt] - iso[r][filt] + A_b-A_r, iso[b][filt] + dm+A_b, color="k")

    x_poly, y_poly = make_polygon(iso, mag_err, dm=dm, A_b=A_b, A_r=A_r, b=b, r=r, iso_width=iso_width)
    plt.plot(x_poly, y_poly, color="k", alpha=0.2)


In [ ]:
def plot_iso_gr(params):
    A_g, A_r, A_i = ana_utils.get_extinction(params.A_V)
    iso = isochrones[params.iso_fe_h][params.iso_log_age]
    plot_iso(iso, dm=params.dm, A_b=A_g, A_r=A_r, mag_err=gr_err, iso_width=params.iso_width)
    
def plot_iso_ri(params):
    A_g, A_r, A_i = ana_utils.get_extinction(params.A_V)
    iso = isochrones[params.iso_fe_h][params.iso_log_age]

    plot_iso(iso, dm=params.dm, A_b=A_r, A_r=A_i, mag_err=ri_err, b="SDSS_r", r="SDSS_i", iso_width=params.iso_width)


In [ ]:
params_best

In [ ]:
df_sorted = df.sort_values("delta")[::-1]
df_sorted = df_sorted[np.isfinite(df.delta)]

In [ ]:
plt.hist(df.significance)

In [ ]:
plt.hist(df.sigma / df.sigma_err, bins=np.linspace(-3, 5, 20))

In [ ]:
df_sorted

In [ ]:
for i in range(10):
    params = retrieve_params(df_sorted.iloc[i, :])
    plot_riso(cat, params)


In [ ]:
for i in range(10):
    params = retrieve_params(df_sorted.iloc[-i-1, :])
    plot_riso(cat, params)

# Sanity checks

In [ ]:
def rand_overdensity(cat, radius, xymax=3):
    xi = np.random.uniform(-xymax+radius, -radius) * np.random.choice([1, -1])
    eta = np.random.uniform(-xymax+radius, -radius) * np.random.choice([1, -1])
    
    area_tot = (2*xymax)**2
    area_cen = np.pi * radius**2
    area_scale = area_cen / (area_tot - area_cen)

    Ntot = count_not_centre(cat, radius, xymax=xymax, xi0=xi, eta0=eta)
    Ncen = count_centre(cat, radius, xi0=xi, eta0=eta)
    
    return counts_to_stats(Ntot, Ncen, area_scale)


In [ ]:
samples = [rand_overdensity(cat, 40/60, xymax=3) for i in  range(10_000)]

In [ ]:
sample_sigma = [x[0] for x in samples]

In [ ]:
sample_err = [x[1] for x in samples]

In [ ]:

plt.hist(sample_sigma)
plt.axvline(inner_overdensity(cat, 40/60)[0], color="C1")

In [ ]:
plt.hist((sample_sigma) / np.array(sample_err), density=True)
x = np.linspace(-5, 5, 1000)
y = 1/np.sqrt(2*np.pi) * np.exp(-x**2 / 2)
plt.plot(x, y)

In [ ]:
cat_test = pd.DataFrame(
    dict(
        xi = np.random.uniform(-3, 3, 2000),
        eta = np.random.uniform(-3, 3, 2000)
    )
)